# Stage 1 — MRI Dataset Exploration, Analysis & Preparation
**MedhaDrishti AI Hackathon — Medical Image Enhancement & Segmentation**

This notebook covers all Stage 1 requirements:
1. Load Brain MRI (BRATS 2020 — T1, T1c, T2, FLAIR) and Spine MRI (Hackathon dataset — T1, T1c, T2, STIR)
2. Compute dataset statistics (per modality, per sample)
3. Compute image property metrics: **Contrast, Complexity, Sharpness, Edge Strength, Noise level, Mean, Std Deviation**
4. Compare sub-modalities (T1 vs T2 vs FLAIR/STIR)
5. Split raw data into training / testing / validation buckets
6. Export a clean statistics CSV — the actual graded deliverable

> Edit the `BRAIN_DATA_DIR` and `SPINE_DATA_DIR` paths in the Config cell to match your Kaggle input paths.

## 0. Setup

In [ ]:
pip install nibabel opencv-python scikit-image pandas matplotlib seaborn --quiet

import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.filters import laplace
from skimage.measure import shannon_entropy

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

print('Libraries loaded OK')

## 1. Config — set your dataset paths here

In [ ]:
# ---- EDIT THESE PATHS ----
BRAIN_DATA_DIR = "Brain_JN_1"
SPINE_DATA_DIR = ""   # offline hackathon dataset folder

OUTPUT_DIR = "/kaggle/working/stage1_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# BRATS naming convention (per-patient folder contains these suffixes)
BRAIN_MODALITY_SUFFIX = {
    'T1': '_t1.nii',
    'T1c': '_t1ce.nii',   # contrast-enhanced T1
    'T2': '_t2.nii',
    'FLAIR': '_flair.nii',
    'SEG': '_seg.nii'     # ground truth mask, BRATS only
}

# Adjust to match your actual hackathon Spine folder naming once you inspect it (see Section 2)
SPINE_MODALITY_KEYWORDS = {
    'T1': ['t1', 'T1'],
    'T1c': ['t1c', 'T1c', 't1ce'],
    'T2': ['t2', 'T2'],
    'STIR': ['stir', 'STIR']
}

print('Config set. Output will be saved to:', OUTPUT_DIR)

## 2. Inventory the raw dataset
First just look at what's actually on disk — folder structure, file counts, naming patterns — before assuming anything.

In [ ]:
def inventory_folder(root_dir, max_show=5):
    if not os.path.isdir(root_dir):
        print(f'[!] Path not found: {root_dir}')
        return []
    patients = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    print(f'Root: {root_dir}')
    print(f'Total patient folders found: {len(patients)}')
    for p in patients[:max_show]:
        files = os.listdir(os.path.join(root_dir, p))
        print(f'  {p}: {files}')
    return patients

print('--- BRAIN ---')
brain_patients = inventory_folder(BRAIN_DATA_DIR)

print()
print('--- SPINE ---')
spine_patients = inventory_folder(SPINE_DATA_DIR)

## 3. Volume loading utilities

In [ ]:
def load_nifti(filepath):
    """Load a .nii/.nii.gz volume, return (numpy array, nibabel image object)."""
    img = nib.load(filepath)
    data = img.get_fdata()
    return data, img

def middle_slice(volume, axis=2):
    """Grab the middle axial slice of a 3D volume for 2D analysis/visualization."""
    idx = volume.shape[axis] // 2
    if axis == 0:
        return volume[idx, :, :]
    elif axis == 1:
        return volume[:, idx, :]
    else:
        return volume[:, :, idx]

def normalize_uint8(slice_2d):
    """Rescale a 2D slice to 0-255 uint8 for OpenCV-based metrics."""
    s = slice_2d.astype(np.float32)
    s -= s.min()
    if s.max() > 0:
        s = s / s.max() * 255.0
    return s.astype(np.uint8)

print('Loading utilities ready')

## 4. Image property metrics
These are the exact properties the problem statement asks for: **Contrast, Complexity, Sharpness, Edge strength, Noise level, Mean, Deviation.**

In [ ]:
def estimate_noise(slice_2d):
    """Fast noise estimate via Laplacian-based method (Immerkaer's formula)."""
    H, W = slice_2d.shape
    M = [[1, -2, 1], [-2, 4, -2], [1, -2, 1]]
    conv = cv2.filter2D(slice_2d.astype(np.float64), -1, np.array(M))
    sigma = np.sum(np.abs(conv))
    sigma = sigma * np.sqrt(0.5 * np.pi) / (6 * (W - 2) * (H - 2))
    return sigma

def compute_image_properties(slice_2d):
    """Compute the full property set for one 2D slice (uint8)."""
    img = slice_2d.astype(np.float32)

    mean_val = np.mean(img)
    std_val = np.std(img)                                  # 'Deviation'
    contrast = img.max() - img.min() if img.size else 0    # simple Michelson-style contrast range
    rms_contrast = std_val / (mean_val + 1e-8)              # normalized contrast
    sharpness = laplace(img).var()                          # Laplacian variance = sharpness
    edge_strength = np.mean(cv2.Canny(slice_2d.astype(np.uint8), 50, 150) > 0)
    complexity = shannon_entropy(slice_2d.astype(np.uint8))  # Entropy as 'complexity'
    noise = estimate_noise(slice_2d)

    return {
        'mean': mean_val,
        'std_dev': std_val,
        'contrast_range': contrast,
        'rms_contrast': rms_contrast,
        'sharpness_laplacian_var': sharpness,
        'edge_strength': edge_strength,
        'complexity_entropy': complexity,
        'noise_level': noise
    }

print('Metric functions ready')

## 5. Batch analysis — Brain MRI (BRATS)
Loops over all patients, all modalities, and computes stats on the middle axial slice of each volume (fast + representative). You can extend to full-volume 3D stats if needed.

In [ ]:
def analyze_brain_dataset(root_dir, patients, suffix_map, max_patients=None):
    records = []
    patients_to_use = patients[:max_patients] if max_patients else patients

    for patient in patients_to_use:
        patient_dir = os.path.join(root_dir, patient)
        for modality, suffix in suffix_map.items():
            if modality == 'SEG':
                continue  # ground truth mask, not an intensity modality
            matches = glob.glob(os.path.join(patient_dir, f'*{suffix}*'))
            if not matches:
                continue
            filepath = matches[0]
            try:
                volume, img_obj = load_nifti(filepath)
                voxel_spacing = img_obj.header.get_zooms()
                slice_2d = middle_slice(volume)
                slice_u8 = normalize_uint8(slice_2d)
                props = compute_image_properties(slice_u8)

                record = {
                    'patient_id': patient,
                    'modality': modality,
                    'shape': volume.shape,
                    'voxel_spacing': voxel_spacing,
                    'filepath': filepath
                }
                record.update(props)
                records.append(record)
            except Exception as e:
                print(f'[!] Failed on {filepath}: {e}')

    return pd.DataFrame(records)

# Set max_patients=None to run on the full set; use a small number first to sanity-check
brain_df = analyze_brain_dataset(BRAIN_DATA_DIR, brain_patients, BRAIN_MODALITY_SUFFIX, max_patients=10)
brain_df.head(10)

## 6. Batch analysis — Spine MRI (Hackathon dataset)

**Important:** Inspect the actual folder/file naming from Section 2 first and adjust `find_spine_modality_file()` below — Spine dataset structure won't match BRATS naming.

In [ ]:
def find_spine_modality_file(patient_dir, keywords):
    """Find a file in patient_dir whose name contains any of the given keywords."""
    all_files = glob.glob(os.path.join(patient_dir, '*.nii*'))
    for f in all_files:
        fname = os.path.basename(f)
        if any(kw in fname for kw in keywords):
            return f
    return None

def analyze_spine_dataset(root_dir, patients, keyword_map, max_patients=None):
    records = []
    patients_to_use = patients[:max_patients] if max_patients else patients

    for patient in patients_to_use:
        patient_dir = os.path.join(root_dir, patient)
        for modality, keywords in keyword_map.items():
            filepath = find_spine_modality_file(patient_dir, keywords)
            if not filepath:
                continue
            try:
                volume, img_obj = load_nifti(filepath)
                voxel_spacing = img_obj.header.get_zooms()
                # Spine is usually read sagittally -- axis=0 for the sagittal slice stack
                slice_2d = middle_slice(volume, axis=0) if volume.ndim == 3 else volume
                slice_u8 = normalize_uint8(slice_2d)
                props = compute_image_properties(slice_u8)

                record = {
                    'patient_id': patient,
                    'modality': modality,
                    'shape': volume.shape,
                    'voxel_spacing': voxel_spacing,
                    'filepath': filepath
                }
                record.update(props)
                records.append(record)
            except Exception as e:
                print(f'[!] Failed on {filepath}: {e}')

    return pd.DataFrame(records)

spine_df = analyze_spine_dataset(SPINE_DATA_DIR, spine_patients, SPINE_MODALITY_KEYWORDS, max_patients=10)
spine_df.head(10)

## 7. Normal vs Pathological split
Both datasets contain 10 normal + 10 pathological samples each. Tag records so stats can be compared healthy vs diseased — pathological cases (tumor, edema, disc herniation) typically show higher complexity/edge strength.

In [ ]:
def tag_condition(patient_id, pathological_ids):
    """pathological_ids: list/set of patient folder names known to be pathological.
    Populate this after inspecting the hackathon dataset's own labeling/folder split."""
    return 'Pathological' if patient_id in pathological_ids else 'Normal'

# EDIT: fill these in once you know which patient folders are normal vs pathological
brain_pathological_ids = set()   # e.g. {'patient_011', 'patient_012', ...}
spine_pathological_ids = set()

if not brain_df.empty:
    brain_df['condition'] = brain_df['patient_id'].apply(lambda x: tag_condition(x, brain_pathological_ids))
if not spine_df.empty:
    spine_df['condition'] = spine_df['patient_id'].apply(lambda x: tag_condition(x, spine_pathological_ids))

print('Condition tagging applied (edit ID sets above once dataset is inspected)')

## 8. Summary statistics table (per modality) — key deliverable

In [ ]:
metric_cols = ['mean', 'std_dev', 'contrast_range', 'rms_contrast',
               'sharpness_laplacian_var', 'edge_strength', 'complexity_entropy', 'noise_level']

def summarize_by_modality(df, label):
    if df.empty:
        print(f'{label}: no data loaded — check dataset path')
        return pd.DataFrame()
    summary = df.groupby('modality')[metric_cols].agg(['mean', 'std']).round(3)
    print(f'--- {label}: summary by modality ---')
    display(summary)
    return summary

brain_summary = summarize_by_modality(brain_df, 'BRAIN MRI')
spine_summary = summarize_by_modality(spine_df, 'SPINE MRI')

## 9. Visual comparison across sub-modalities

In [ ]:
def plot_metric_comparison(df, label):
    if df.empty:
        return
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()
    for i, metric in enumerate(metric_cols):
        sns.boxplot(data=df, x='modality', y=metric, ax=axes[i])
        axes[i].set_title(metric)
        axes[i].tick_params(axis='x', rotation=45)
    fig.suptitle(f'{label}: Image property comparison across modalities', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{label.lower()}_modality_comparison.png'), dpi=150)
    plt.show()

plot_metric_comparison(brain_df, 'Brain')
plot_metric_comparison(spine_df, 'Spine')

## 10. Sample slice visualization (qualitative check)

In [ ]:
def show_patient_modalities(root_dir, patient, suffix_or_keyword_map, is_brats=True):
    patient_dir = os.path.join(root_dir, patient)
    fig, axes = plt.subplots(1, len(suffix_or_keyword_map), figsize=(4 * len(suffix_or_keyword_map), 4))
    if len(suffix_or_keyword_map) == 1:
        axes = [axes]

    for ax, (modality, key) in zip(axes, suffix_or_keyword_map.items()):
        if modality == 'SEG':
            continue
        filepath = None
        if is_brats:
            matches = glob.glob(os.path.join(patient_dir, f'*{key}*'))
            filepath = matches[0] if matches else None
        else:
            filepath = find_spine_modality_file(patient_dir, key)

        if filepath:
            volume, _ = load_nifti(filepath)
            slice_2d = middle_slice(volume, axis=2 if is_brats else 0) if volume.ndim == 3 else volume
            ax.imshow(slice_2d.T, cmap='gray', origin='lower')
        ax.set_title(modality)
        ax.axis('off')

    plt.suptitle(f'Patient: {patient}')
    plt.tight_layout()
    plt.show()

if brain_patients:
    show_patient_modalities(BRAIN_DATA_DIR, brain_patients[0], BRAIN_MODALITY_SUFFIX, is_brats=True)

if spine_patients:
    show_patient_modalities(SPINE_DATA_DIR, spine_patients[0], SPINE_MODALITY_KEYWORDS, is_brats=False)

## 11. Train / Test / Validation split
Simple, reproducible 70/15/15 split at the **patient level** (never split a patient's own scans across sets — that would leak information).

In [ ]:
from sklearn.model_selection import train_test_split

def make_split(patients, train_size=0.7, val_size=0.15, seed=42):
    if len(patients) < 3:
        print('Too few patients to split meaningfully — using all as train for now')
        return patients, [], []
    train, temp = train_test_split(patients, train_size=train_size, random_state=seed)
    rel_val = val_size / (1 - train_size)
    val, test = train_test_split(temp, train_size=rel_val, random_state=seed)
    return train, val, test

brain_train, brain_val, brain_test = make_split(brain_patients)
spine_train, spine_val, spine_test = make_split(spine_patients)

print('Brain  -> train:', len(brain_train), '| val:', len(brain_val), '| test:', len(brain_test))
print('Spine  -> train:', len(spine_train), '| val:', len(spine_val), '| test:', len(spine_test))

split_records = []
for name, split in [('train', brain_train), ('val', brain_val), ('test', brain_test)]:
    for p in split:
        split_records.append({'organ': 'brain', 'patient_id': p, 'split': name})
for name, split in [('train', spine_train), ('val', spine_val), ('test', spine_test)]:
    for p in split:
        split_records.append({'organ': 'spine', 'patient_id': p, 'split': name})

split_df = pd.DataFrame(split_records)
split_df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_split.csv'), index=False)
split_df.head(10)

## 12. Export final Stage 1 deliverables

In [ ]:
if not brain_df.empty:
    brain_df.to_csv(os.path.join(OUTPUT_DIR, 'brain_mri_image_properties.csv'), index=False)
if not spine_df.empty:
    spine_df.to_csv(os.path.join(OUTPUT_DIR, 'spine_mri_image_properties.csv'), index=False)

print('Saved deliverables to:', OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

## Notes / Next steps

- **Fix the paths** in Section 1 to your actual Kaggle input directories once BRATS and the Spine dataset are attached.
- **Inspect Section 2 output carefully** for the Spine dataset — its file naming almost certainly won't match `SPINE_MODALITY_KEYWORDS` out of the box. Update the keyword lists once you see real filenames.
- **Fill in `brain_pathological_ids` / `spine_pathological_ids`** in Section 7 once you know which patient folders are pathological vs normal (organizers usually separate these into different sub-folders, e.g. `Normal/` and `Pathological/`).
- Currently metrics are computed on the **middle 2D slice** per volume for speed. If graders expect full-3D property assessment, extend `compute_image_properties` to operate over the whole volume (slice-by-slice averaging or full 3D Laplacian).
- This notebook produces exactly what Stage 1 asks for: dataset statistics, per-modality image property assessment (contrast, complexity, sharpness, edge strength, noise, mean, deviation), sub-modality comparison, and a train/val/test split — ready to feed into Stage 2 preprocessing.